# GRU Autoencoder IDS — Privacy–Utility Tradeoff
Source-of-truth notebook. All model logic lives in `src/`; this notebook orchestrates training, evaluation, and the final comparison table + plot.

**Run from the repo root:** `jupyter notebook notebooks/workflow.ipynb`

## 0. Config

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root on path

DATA_DIR   = "../data/CARLA_processed"
MODELS_DIR = "models"
FIGURES_DIR = "figures"
EPOCHS     = 30
BATCH_SIZE = 256

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Data loading & EDA

In [ ]:
from src.data import load_sessions, split_sessions, make_dataset, FEATURES

sessions = load_sessions(DATA_DIR)
print(f"Sessions loaded: {len(sessions)}")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Total rows: {sum(len(s) for s in sessions):,}")

# Per-session row counts
import pandas as pd
pd.Series([len(s) for s in sessions]).describe()

In [ ]:
import matplotlib.pyplot as plt

sample = sessions[0]
fig, axes = plt.subplots(3, 3, figsize=(12, 7))
for ax, col in zip(axes.flat, FEATURES):
    ax.plot(sample[col].values[:500], linewidth=0.6)
    ax.set_title(col, fontsize=9)
    ax.set_xticks([])
fig.suptitle("First 500 timesteps — session 0", fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
train_s, val_s, test_s = split_sessions(sessions)
print(f"Train: {len(train_s)} sessions  Val: {len(val_s)}  Test: {len(test_s)}")

train_ds, scaler = make_dataset(train_s, fit_scaler=True)
val_ds,   _      = make_dataset(val_s,   scaler=scaler)
test_ds,  _      = make_dataset(test_s,  scaler=scaler)

print(f"Train windows: {len(train_ds):,}")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")

## 2. Baseline GRU Autoencoder

In [ ]:
from src.model import GRUAutoencoder
from src.train import train

BASELINE_PATH = f"{MODELS_DIR}/baseline.pth"

baseline_model = GRUAutoencoder(input_size=len(FEATURES), hidden_size=64, num_layers=2)

if os.path.exists(BASELINE_PATH):
    print("Loading existing baseline checkpoint …")
    baseline_model.load_state_dict(torch.load(BASELINE_PATH, map_location=DEVICE, weights_only=True))
    baseline_model = baseline_model.to(DEVICE)
else:
    print("Training baseline … (run this cell in the terminal for speed)")
    print(f"  python -m src.compare --data_dir {DATA_DIR} --out_dir notebooks --epochs {EPOCHS}")
    baseline_model = train(
        baseline_model, train_ds, val_ds,
        save_path=BASELINE_PATH,
        epochs=EPOCHS, batch_size=BATCH_SIZE, device=DEVICE,
    )

## 3. Baseline evaluation

In [ ]:
from src.evaluate import reconstruction_errors, make_attacks, compute_metrics

baseline_metrics = compute_metrics(baseline_model, test_ds, val_dataset=val_ds, device=DEVICE)
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Reconstruction error distribution: normal vs attacked
import numpy as np

normal_errs = reconstruction_errors(baseline_model, test_ds, device=DEVICE)
attacked_ds, _ = make_attacks(test_ds)
attack_errs = reconstruction_errors(baseline_model, attacked_ds, device=DEVICE)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(normal_errs, bins=80, alpha=0.6, label="Normal", color="steelblue")
ax.hist(attack_errs, bins=80, alpha=0.6, label="Attack", color="tomato")
ax.axvline(baseline_metrics["threshold"], color="black", linestyle="--", label="Threshold")
ax.set_xlabel("Reconstruction MSE")
ax.set_ylabel("Count")
ax.set_title("Baseline: Reconstruction Error Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Full privacy–utility sweep

This runs all three perturbation families. For large datasets, **run in the terminal** instead:
```bash
python -m src.compare --data_dir data/CARLA_processed --out_dir notebooks --epochs 30
```

In [ ]:
from src.compare import run as run_comparison

results_df = run_comparison(
    data_dir=DATA_DIR,
    out_dir=".",   # saves relative to notebooks/
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

## 5. Results

In [ ]:
results_df

In [ ]:
from IPython.display import Image
Image(f"{FIGURES_DIR}/privacy_utility.png")